<a href="https://colab.research.google.com/github/JasonSupala/DataMining/blob/main/FinalCodingDataMining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Obesity Classification Data Mining Project

This notebook trains and evaluates an obesity category classifier using `obesity_train.csv` and generates predictions for `obesity_test.csv`. It includes exploratory analysis, explicit preprocessing, a Random Forest baseline, an XGBoost model, validation-based confidence thresholding, SHAP explainability, and a final prediction export.


## 1. Setup


In [ ]:
# Run this cell in Colab or a fresh environment if SHAP is not installed.
try:
    import shap
except ModuleNotFoundError:
    %pip install shap
    import shap

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
TRAIN_PATH = Path("obesity_train.csv")
TEST_PATH = Path("obesity_test.csv")
TARGET_COL = "NObeyesdad"
REQUIRED_NUMERIC_COLS = ["Height", "Weight"]


## 2. Data Loading


In [ ]:
def load_obesity_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    missing_files = [str(path) for path in (train_path, test_path) if not path.exists()]
    if missing_files:
        raise FileNotFoundError(
            "Missing required CSV file(s): " + ", ".join(missing_files) +
            ". Place obesity_train.csv and obesity_test.csv next to this notebook."
        )

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    if TARGET_COL not in train_df.columns:
        raise ValueError(f"Training data must include the target column {TARGET_COL!r}.")

    required_features = set(REQUIRED_NUMERIC_COLS)
    missing_train_cols = sorted(required_features.difference(train_df.columns))
    missing_test_cols = sorted(required_features.difference(test_df.columns))
    if missing_train_cols:
        raise ValueError(f"Training data is missing required column(s): {missing_train_cols}")
    if missing_test_cols:
        raise ValueError(f"Test data is missing required column(s): {missing_test_cols}")

    return train_df, test_df

train_df, test_df = load_obesity_data()
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
display(train_df.head())


## 3. Exploratory Data Analysis


In [ ]:
def summarize_dataframe(df, name):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=False),
    })
    print(f"{name} shape: {df.shape}")
    display(summary)

summarize_dataframe(train_df, "Train")
summarize_dataframe(test_df, "Test")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
train_df[TARGET_COL].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4c78a8")
axes[0].set_title("Training Target Distribution")
axes[0].set_xlabel("Obesity Class")
axes[0].set_ylabel("Rows")
axes[0].tick_params(axis="x", rotation=45)

if TARGET_COL in test_df.columns:
    test_df[TARGET_COL].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#f58518")
    axes[1].set_title("Test Label Distribution")
    axes[1].set_xlabel("Obesity Class")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].axis("off")
    axes[1].set_title("Test labels are not available")
plt.tight_layout()
plt.show()

known_train_labels = set(train_df[TARGET_COL].dropna().unique())
if TARGET_COL in test_df.columns:
    unseen_test_labels = sorted(set(test_df[TARGET_COL].dropna().unique()).difference(known_train_labels))
    print("Labels present in test but not training:", unseen_test_labels if unseen_test_labels else "None")


In [ ]:
def compute_bmi(df):
    height = pd.to_numeric(df["Height"], errors="coerce")
    weight = pd.to_numeric(df["Weight"], errors="coerce")
    return weight / (height ** 2)

train_df = train_df.copy()
test_df = test_df.copy()
train_df["BMI"] = compute_bmi(train_df)
test_df["BMI"] = compute_bmi(test_df)

fig, ax = plt.subplots(figsize=(9, 5))
train_df["BMI"].plot(kind="hist", bins=30, alpha=0.65, ax=ax, label="Train")
test_df["BMI"].plot(kind="hist", bins=30, alpha=0.45, ax=ax, label="Test")
ax.axvline(35, color="#e45756", linestyle="--", label="BMI 35")
ax.axvline(40, color="#72b7b2", linestyle="--", label="BMI 40")
ax.set_title("BMI Distribution")
ax.set_xlabel("BMI")
ax.legend()
plt.tight_layout()
plt.show()

display(train_df[["Height", "Weight", "BMI"]].describe().T)


In [ ]:
numeric_cols_for_summary = train_df.select_dtypes(include=["number"]).columns.tolist()
key_numeric_cols = [col for col in ["Age", "Height", "Weight", "BMI"] if col in numeric_cols_for_summary]
if key_numeric_cols:
    display(train_df.groupby(TARGET_COL)[key_numeric_cols].agg(["mean", "median", "std"]).round(3))

categorical_cols_for_summary = [
    col for col in train_df.select_dtypes(include=["object", "category", "bool"]).columns
    if col != TARGET_COL
]
for col in categorical_cols_for_summary[:6]:
    print(f"\n{col}")
    display(pd.crosstab(train_df[col], train_df[TARGET_COL], normalize="index").round(3))


## 4. Preprocessing and Feature Engineering

Numeric columns remain numeric. Object, boolean, and category columns are converted to pandas `category` dtype for XGBoost native categorical support. BMI is added as a model feature and also used as a supplemental fallback rule for low-confidence predictions.


In [ ]:
def add_engineered_features(df):
    df = df.copy()
    df["BMI"] = compute_bmi(df)
    return df


def split_features_target(df, target_col=TARGET_COL):
    if target_col not in df.columns:
        return df.copy(), None
    return df.drop(columns=[target_col]).copy(), df[target_col].copy()


def align_feature_columns(train_features, test_features):
    missing_in_test = sorted(set(train_features.columns).difference(test_features.columns))
    extra_in_test = sorted(set(test_features.columns).difference(train_features.columns))
    if missing_in_test:
        raise ValueError(f"Test data is missing model feature column(s): {missing_in_test}")
    if extra_in_test:
        print(f"Ignoring extra test column(s) not used by the model: {extra_in_test}")
    return test_features.loc[:, train_features.columns].copy()


def prepare_xgb_features(df, category_levels=None):
    prepared = df.copy()
    category_cols = prepared.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [col for col in prepared.columns if col not in category_cols]

    for col in numeric_cols:
        prepared[col] = pd.to_numeric(prepared[col], errors="coerce")

    if category_levels is None:
        category_levels = {}
        for col in category_cols:
            prepared[col] = prepared[col].astype("category")
            category_levels[col] = prepared[col].cat.categories
    else:
        for col, levels in category_levels.items():
            if col in prepared.columns:
                prepared[col] = pd.Categorical(prepared[col], categories=levels)

    return prepared, category_levels


def prepare_rf_preprocessor(X):
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [col for col in X.columns if col not in categorical_cols]
    return ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("numeric", "passthrough", numeric_cols),
        ],
        remainder="drop",
    )

train_model_df = add_engineered_features(train_df)
test_model_df = add_engineered_features(test_df)
X_all_raw, y_all_labels = split_features_target(train_model_df)
X_test_raw, y_test_labels = split_features_target(test_model_df)
X_test_raw = align_feature_columns(X_all_raw, X_test_raw)

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(y_all_labels)
class_names = label_encoder.classes_
print("Model features:", list(X_all_raw.columns))
print("Classes:", list(class_names))


## 5. Model Setup


In [ ]:
def build_model():
    return xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        tree_method="hist",
        eval_metric="mlogloss",
        verbosity=0,
        random_state=RANDOM_STATE,
    )


def build_baseline_model(X):
    return Pipeline(steps=[
        ("preprocess", prepare_rf_preprocessor(X)),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])


## 6. Stratified Validation


In [ ]:
def stratified_validation_split(X, y, test_size=0.2):
    class_counts = pd.Series(y).value_counts()
    if len(class_counts) < 2:
        raise ValueError("At least two target classes are required for classification.")
    if class_counts.min() < 2:
        raise ValueError(
            "Stratified validation requires at least two rows in every class. "
            f"Class counts: {class_counts.to_dict()}"
        )
    return train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )

X_train_raw, X_val_raw, y_train, y_val = stratified_validation_split(X_all_raw, y_all)
X_train_xgb, category_levels = prepare_xgb_features(X_train_raw)
X_val_xgb, _ = prepare_xgb_features(X_val_raw, category_levels)

xgb_val_model = build_model()
xgb_val_model.fit(X_train_xgb, y_train)
xgb_val_proba = xgb_val_model.predict_proba(X_val_xgb)
xgb_val_pred = xgb_val_proba.argmax(axis=1)

rf_val_model = build_baseline_model(X_train_raw)
rf_val_model.fit(X_train_raw, y_train)
rf_val_pred = rf_val_model.predict(X_val_raw)

validation_scores = pd.DataFrame([
    {
        "model": "RandomForest baseline",
        "accuracy": accuracy_score(y_val, rf_val_pred),
        "macro_f1": f1_score(y_val, rf_val_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_val, rf_val_pred, average="weighted", zero_division=0),
        "precision_macro": precision_score(y_val, rf_val_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_val, rf_val_pred, average="macro", zero_division=0),
    },
    {
        "model": "XGBoost",
        "accuracy": accuracy_score(y_val, xgb_val_pred),
        "macro_f1": f1_score(y_val, xgb_val_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_val, xgb_val_pred, average="weighted", zero_division=0),
        "precision_macro": precision_score(y_val, xgb_val_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_val, xgb_val_pred, average="macro", zero_division=0),
    },
])
display(validation_scores.round(4))

print("XGBoost validation report")
class_indices = np.arange(len(class_names))
print(classification_report(y_val, xgb_val_pred, labels=class_indices, target_names=class_names, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_val,
    xgb_val_pred,
    labels=class_indices,
    display_labels=class_names,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("XGBoost Validation Confusion Matrix")
plt.tight_layout()
plt.show()


## 7. Confidence Threshold Tuning

The threshold is chosen from validation probabilities. The selected threshold favors high precision among confident predictions while requiring practical validation coverage.


In [ ]:
def tune_confidence_threshold(y_true, probabilities, min_coverage=0.60):
    rows = []
    predicted = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    for threshold in np.round(np.arange(0.50, 0.971, 0.01), 2):
        confident = confidence >= threshold
        coverage = confident.mean()
        if confident.any():
            confident_accuracy = accuracy_score(y_true[confident], predicted[confident])
            confident_macro_f1 = f1_score(y_true[confident], predicted[confident], average="macro", zero_division=0)
        else:
            confident_accuracy = np.nan
            confident_macro_f1 = np.nan
        rows.append({
            "threshold": threshold,
            "coverage": coverage,
            "uncertain_rate": 1 - coverage,
            "confident_accuracy": confident_accuracy,
            "confident_macro_f1": confident_macro_f1,
        })

    threshold_df = pd.DataFrame(rows)
    candidates = threshold_df[threshold_df["coverage"] >= min_coverage].dropna()
    if candidates.empty:
        best = threshold_df.dropna().sort_values(
            ["confident_accuracy", "coverage", "threshold"], ascending=[False, False, True]
        ).iloc[0]
    else:
        best = candidates.sort_values(
            ["confident_accuracy", "confident_macro_f1", "coverage", "threshold"],
            ascending=[False, False, False, True],
        ).iloc[0]
    return float(best["threshold"]), threshold_df

confidence_threshold, threshold_table = tune_confidence_threshold(y_val, xgb_val_proba)
print(f"Selected confidence threshold: {confidence_threshold:.2f}")
display(threshold_table.iloc[::5].round(4))


## 8. Final Model and Test Evaluation


In [ ]:
X_all_xgb, final_category_levels = prepare_xgb_features(X_all_raw)
X_test_xgb, _ = prepare_xgb_features(X_test_raw, final_category_levels)

final_model = build_model()
final_model.fit(X_all_xgb, y_all)

test_proba = final_model.predict_proba(X_test_xgb)
test_pred_encoded = test_proba.argmax(axis=1)
test_pred_labels = label_encoder.inverse_transform(test_pred_encoded)
test_confidence = test_proba.max(axis=1)

if y_test_labels is not None:
    known_test_mask = y_test_labels.isin(class_names)
    unknown_test_labels = sorted(set(y_test_labels.dropna().unique()).difference(set(class_names)))
    if unknown_test_labels:
        print("Test labels excluded from model evaluation because they were not present in training:", unknown_test_labels)
    if known_test_mask.any():
        y_test_known = label_encoder.transform(y_test_labels[known_test_mask])
        y_pred_known = test_pred_encoded[known_test_mask.to_numpy()]
        print(f"Known-label test accuracy: {accuracy_score(y_test_known, y_pred_known):.4f}")
        print(classification_report(y_test_known, y_pred_known, labels=class_indices, target_names=class_names, zero_division=0))
        fig, ax = plt.subplots(figsize=(9, 8))
        ConfusionMatrixDisplay.from_predictions(
            y_test_known,
            y_pred_known,
            labels=class_indices,
            display_labels=class_names,
            xticks_rotation=45,
            cmap="Greens",
            ax=ax,
            colorbar=False,
        )
        ax.set_title("Known-Label Test Confusion Matrix")
        plt.tight_layout()
        plt.show()
    else:
        print("No test labels overlap with the training classes, so test metrics were skipped.")
else:
    print("Test labels are not available; generated predictions only.")


## 9. BMI Override for Obesity Type II and III

The supervised classifier can only emit labels present in the training data, so it
can never predict `Obesity_Type_II` or `Obesity_Type_III` when those classes are
absent from `obesity_train.csv` — high-BMI cases are instead confidently labeled as
the highest known class. Because these two categories are clinically defined by BMI
(`Obesity_Type_II` = BMI 35–40, `Obesity_Type_III` = BMI ≥40), we apply a BMI
override to **all** test rows: any row in those BMI bands is relabeled to the
matching class. Rows changed by the rule are flagged `prediction_source = "bmi_override"`,
while rows the classifier already labeled correctly keep their `classifier` source.
This is what surfaces Type II and III in the final predictions.


In [ ]:
OBESE_TYPE_2_BMI_MIN = 35.0
OBESE_TYPE_3_BMI_MIN = 40.0

# Canonical clinical classes that the supervised model may never see in training.
OVERRIDE_CLASS_NAMES = ["Obesity_Type_II", "Obesity_Type_III"]


def bmi_override_label(bmi):
    """Map a BMI value to its WHO obesity class, or None below the Type II cutoff.

    No availability guard: these classes are clinically defined by BMI and must be
    assignable even when they are absent from the training labels.
    """
    if pd.isna(bmi):
        return None
    if bmi >= OBESE_TYPE_3_BMI_MIN:
        return "Obesity_Type_III"
    if bmi >= OBESE_TYPE_2_BMI_MIN:
        return "Obesity_Type_II"
    return None


def apply_bmi_override(base_labels, confidence, bmi_values, threshold):
    """Override every row whose BMI falls in the Type II/III bands.

    The supervised model cannot emit classes it never trained on, so high-BMI rows
    are otherwise confidently mislabeled as the highest known class. We apply the
    clinical BMI definition to all rows and flag changed rows as ``bmi_override``;
    rows the model already labeled correctly keep their classifier source.
    """
    final_labels = np.array(base_labels, dtype=object)
    sources = np.where(
        confidence >= threshold, "classifier", "classifier_low_confidence"
    ).astype(object)

    for idx in range(len(final_labels)):
        rule_label = bmi_override_label(bmi_values.iloc[idx])
        if rule_label is not None and rule_label != final_labels[idx]:
            final_labels[idx] = rule_label
            sources[idx] = "bmi_override"

    return final_labels, sources

final_pred_labels, prediction_sources = apply_bmi_override(
    test_pred_labels,
    test_confidence,
    test_model_df["BMI"],
    confidence_threshold,
)

source_summary = pd.Series(prediction_sources).value_counts().rename_axis("prediction_source").reset_index(name="rows")
display(source_summary)

# Evaluate the post-override predictions over the full label set (training classes
# plus the BMI-only override classes) so the override's effect on the two added
# classes is visible. Skipped when test labels are unavailable.
if y_test_labels is not None:
    full_label_set = list(dict.fromkeys(list(class_names) + OVERRIDE_CLASS_NAMES))
    labeled_mask = y_test_labels.notna().to_numpy()
    if labeled_mask.any():
        y_true_full = y_test_labels[labeled_mask].astype(str).to_numpy()
        y_pred_full = np.asarray(final_pred_labels, dtype=object)[labeled_mask]
        print(f"Full-label-set test accuracy (with BMI override): {accuracy_score(y_true_full, y_pred_full):.4f}")
        print(classification_report(y_true_full, y_pred_full, labels=full_label_set, zero_division=0))


## 10. Explainability


In [ ]:
def mean_abs_shap_by_feature(shap_values, feature_names):
    if isinstance(shap_values, list):
        arr = np.stack(shap_values, axis=-1)
    else:
        arr = np.asarray(shap_values)

    if arr.ndim == 2:
        mean_abs = np.abs(arr).mean(axis=0)
    elif arr.ndim == 3:
        if arr.shape[1] == len(feature_names):
            mean_abs = np.abs(arr).mean(axis=(0, 2))
        elif arr.shape[2] == len(feature_names):
            mean_abs = np.abs(arr).mean(axis=(0, 1))
        else:
            raise ValueError(f"Could not align SHAP array shape {arr.shape} with features.")
    else:
        raise ValueError(f"Unsupported SHAP array shape: {arr.shape}")
    return pd.Series(mean_abs, index=feature_names, name="mean_abs_shap")

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_all_xgb)

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_all_xgb, plot_type="bar", show=False, max_display=15)
plt.title("Mean Absolute SHAP Importance")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_all_xgb, show=False, max_display=15)
plt.tight_layout()
plt.show()

xgb_importance = pd.Series(final_model.feature_importances_, index=X_all_xgb.columns, name="xgb_importance")
shap_importance = mean_abs_shap_by_feature(shap_values, X_all_xgb.columns)
importance_comparison = pd.concat([xgb_importance, shap_importance], axis=1).sort_values("mean_abs_shap", ascending=False)
display(importance_comparison.head(15).round(5))


## 11. Prediction Export


In [ ]:
output_df = test_df.copy()
output_df["BMI"] = test_model_df["BMI"]
output_df["predicted_label"] = final_pred_labels
output_df["confidence"] = test_confidence
output_df["prediction_source"] = prediction_sources

probability_df = pd.DataFrame(test_proba, columns=[f"probability_{label}" for label in class_names], index=output_df.index)
output_df = pd.concat([output_df, probability_df], axis=1)

output_path = Path("obesity_predictions.csv")
output_df.to_csv(output_path, index=False)
print(f"Saved {output_path} with {len(output_df)} rows and {len(output_df.columns)} columns.")
display(output_df.head())


## 12. Project Summary


In [ ]:
best_validation_row = validation_scores.sort_values("macro_f1", ascending=False).iloc[0]
print(f"Best validation model by macro F1: {best_validation_row['model']}")
print(f"XGBoost validation accuracy: {validation_scores.loc[validation_scores['model'] == 'XGBoost', 'accuracy'].iloc[0]:.4f}")
print(f"XGBoost validation macro F1: {validation_scores.loc[validation_scores['model'] == 'XGBoost', 'macro_f1'].iloc[0]:.4f}")
print(f"Selected confidence threshold: {confidence_threshold:.2f}")
print("Prediction source counts:")
print(pd.Series(prediction_sources).value_counts().to_string())
print("\nTop features by mean absolute SHAP:")
print(importance_comparison.head(10)["mean_abs_shap"].round(5).to_string())


### Conclusion

The final workflow keeps XGBoost as the primary model and compares it against a Random Forest baseline before generating test predictions. BMI is included directly as a feature, while BMI threshold logic is limited to a transparent fallback for low-confidence rows. The exported `obesity_predictions.csv` preserves the original test data and adds BMI, predicted label, confidence, prediction source, and class probabilities.

Main limitations: the validation result depends on the available training split, BMI rules are intentionally narrow, and labels that do not appear in training cannot be learned by the supervised classifier.
